# Prepare data

In [ ]:
import pickle
import os

all_data = []
base_dir = "dream/small_model_train/train_data"
for file_name in os.listdir(base_dir):
    with open(f"{base_dir}/{file_name}", "rb") as file:
        try:
            data = pickle.load(file)
        except:
            print(file_name)
        all_data += data
num_block = len(([d['step'] for d in all_data if d['step'] == 1]))
print(f"Average steps: {len(all_data) / num_block}")
print(f"Total data count: {len(all_data)}")
print(all_data[0])

In [ ]:
import random

random.shuffle(all_data)
data_num = len(all_data)
print(all_data[0])

In [ ]:
split1 = int(data_num * 0.8)
split2 = split1 + int(data_num * 0.1)
train_data = all_data[:split1]
val_data = all_data[split1:split2]
test_data = all_data[split2:]
print(len(train_data), len(val_data), len(test_data))

## Load data

In [ ]:
import pandas as pd

In [ ]:
df = pd.DataFrame(train_data)

In [ ]:
len(df)

In [ ]:
import torch
import numpy as np
device = 'cpu'

X = np.stack(df['confidence'].values)
y = np.stack(df['is_correct'].apply(lambda x: [int(b) for b in x]))

X = torch.tensor(X, dtype=torch.float32, device=device)
y = torch.tensor(y, dtype=torch.float32, device=device)

In [ ]:
df_val = pd.DataFrame(val_data)
X_val = np.stack(df_val['confidence'].values)
y_val = np.stack(df_val['is_correct'].apply(lambda x: [int(b) for b in x]))

X_val = torch.tensor(X_val, dtype=torch.float32, device=device)
y_val = torch.tensor(y_val, dtype=torch.float32, device=device)

In [ ]:
import torch.nn as nn
from dream.model.small_model import LogisticRegression

model = LogisticRegression(input_dim=32)
model.to(device)

In [ ]:
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.003)

# Training

In [ ]:
import matplotlib.pyplot as plt
import torch
import numpy as np

losses = []
val_losses = []
false_postives = []

for epoch in range(40000):
    optimizer.zero_grad()
    logits = model(X)
    loss = loss_fn(logits, y)
    loss.backward()
    optimizer.step()

    current_loss = loss.item()
    losses.append(current_loss)

    model.eval()
    with torch.no_grad():
        val_logits = model(X_val)
        val_loss = loss_fn(val_logits, y_val).item()
        val_losses.append(val_loss)
        probs = torch.sigmoid(val_logits)
        preds = (probs > 0.9).float()
        false_pos = ((preds > y_val).sum() / preds.sum()).item()
        false_postives.append(false_pos)

    if epoch % 500 == 0:
        print(f"Epoch {epoch}: loss={loss:.4f}, Val_loss={val_loss:.4f}, False_postive={false_pos:.4f}")


# Plot traning stats & save model

In [ ]:
plt.figure(figsize=(6, 6))
plt.plot(losses, label='Training Loss', color='#0066CC')
plt.plot(val_losses, label='Validation Loss', color='red')

plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.gca().set_facecolor('#f8f8f8')


plt.savefig('learning_curve.png', dpi=300, bbox_inches='tight')
plt.show()

# --- Figure 2: False Positive Rate ---
plt.figure(figsize=(6, 6))
plt.plot(false_postives, label='False Positive Rate', color='green')

plt.xlabel('Epoch')
plt.ylabel('False Positive Rate')
plt.legend()
plt.grid(True)
plt.title('False Positive Rate over Epochs')

plt.show()

In [ ]:
# model.load_state_dict(torch.load("dream/layer_2_flan.pth"))
df_test = pd.DataFrame(test_data)
X_test = np.stack(df_test['confidence'].values)
y_test = np.stack(df_test['is_correct'].apply(lambda x: [int(b) for b in x]))

X_test = torch.tensor(X_test, dtype=torch.float32, device=device)
y_test = torch.tensor(y_test, dtype=torch.float32, device=device)

In [ ]:
model.to(X_test.device)
th = 0.9
with torch.no_grad():
    logits = model(X_test)
    probs = torch.sigmoid(logits)
    preds = (probs > th).float()
    acc = (preds == y_test).float().mean().item()
    false_neg = ((preds < y_test).sum() / (preds.numel() - preds.sum())).item()
    false_pos = ((preds > y_test).sum() / preds.sum()).item()
    print("threshold:", th, "Accuracy:", acc, "False Negative: ", false_neg, "False Positive: ", false_pos)

In [ ]:
torch.save(model.to('cpu').state_dict(), 'layer_2_flan.pth')